# Transform DFA Superdataset to RDF  
* Uses `root/owl/superdataset.ttl` to transform the DFA Superdataset to RDF.
* Run `make superdataset-owl` from the root directory to build.
* Run `make superdataset-data.ttl` to create the rdf data: `root/notebooks/DFA-data/superdataset-data.ttl`

**Dependencies**
* python >= 3.11
* [pandas](https://pypi.org/project/pandas/): load the superdataset CSV and show query results as tables
* [rdflib](https://pypi.org/project/rdflib/): load `superdataset-data.ttl` and run SPARQL queries on it
* [SPARQLWrapper](https://pypi.org/project/SPARQLWrapper/): query a SPARQL endpoint with `sparql_df`
* [rdflib-endpoint](https://pypi.org/project/rdflib-endpoint/): serve the data as a SPARQL endpoint (imported in the first code cell)

```
pip install pandas rdflib SPARQLWrapper
```

Building the RDF files with `make` also needs [linkml](https://pypi.org/project/linkml/), [click](https://pypi.org/project/click/) and [ROBOT](https://robot.obolibrary.org/).

In [1]:
import pandas as pd
from itertools import islice
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Literal, URIRef, BNode
from SPARQLWrapper import SPARQLWrapper, JSON
from decimal import Decimal

## helper functions

In [2]:
def head(path, start=1, stop=None):
    """Return lines start..stop of a file (1-based, inclusive) as a string.

    Defaults: start at the first line; stop 5 lines later.
    """
    if start < 1:
        raise ValueError("start must be >= 1")
    if stop is None:
        stop = start + 4
    if stop < start:
        raise ValueError("stop must be >= start")
    with open(path) as f:
        return "".join(islice(f, start - 1, stop))

In [3]:
def sparql_df(query, source="http://127.0.0.1:8000/", to_python=True):
    """Run a SPARQL SELECT query and return the results as a pandas DataFrame.

    Args:
        query: SPARQL SELECT query.
        source: an rdflib Graph, or the URL of a SPARQL endpoint.
        to_python: convert literals to Python values (e.g., xsd:integer -> int);
            if False, every value is returned as a string.
    """
    def convert(term):
        if term is None:
            return None
        if to_python and isinstance(term, Literal):
            value = term.toPython()
            return float(value) if isinstance(value, Decimal) else value
        return str(term)

    def to_term(b):
        # JSON result binding -> rdflib term
        if b is None:
            return None
        if b["type"] == "uri":
            return URIRef(b["value"])
        if b["type"] == "bnode":
            return BNode(b["value"])
        return Literal(b["value"], lang=b.get("xml:lang"), datatype=b.get("datatype"))

    if isinstance(source, Graph):
        res = source.query(query)
        cols = [str(v) for v in res.vars]
        rows = [[convert(term) for term in row] for row in res]
    else:
        sw = SPARQLWrapper(source)
        sw.setQuery(query)
        sw.setReturnFormat(JSON)
        res = sw.query().convert()
        cols = res["head"]["vars"]
        rows = [[convert(to_term(b.get(c))) for c in cols] for b in res["results"]["bindings"]]

    df = pd.DataFrame(rows, columns=cols)
    # use nullable types so integer columns with missing values stay integers
    return df.convert_dtypes() if to_python else df

In [4]:
df = pd.read_csv('DFA-data/merged_dfa_data_cleaned_2026_09_30.csv', low_memory=False)
df.shape

(10163, 65)

In [5]:
df.Participant_id.nunique()

10163

In [6]:
df.columns

Index(['Participant_id', 'Participant_id_original', 'Last_Dental_Visit', 'Sex',
       'Age', 'agegrp', 'Ethnicity_new', 'Race', 'dfs1', 'dfs2', 'dfs3',
       'dfs4', 'dfs5', 'dfs6', 'dfs7', 'dfs8', 'dfs9', 'dfs10', 'dfs11',
       'dfs12', 'dfs13', 'dfs14', 'dfs15', 'dfs16', 'dfs17', 'dfs18', 'dfs19',
       'dfs20', 'idaf1a', 'idaf1b', 'idaf1c', 'idaf1d', 'idaf1e', 'idaf1f',
       'idaf1g', 'idaf1h', 'idaf2a', 'idaf2b', 'idaf2c', 'idaf2d', 'idaf2e',
       'idaf3a', 'idaf3b', 'idaf3c', 'idaf3d', 'idaf3e', 'idaf3f', 'idaf3g',
       'idaf3h', 'idaf3i', 'idaf3j', 'mdas1', 'mdas2', 'mdas3', 'mdas4',
       'mdas5', 'source', 'count_missing_dfs', 'dfs_flag', 'idaf_flag',
       'mdas_flag', 'dfs_total_score', 'dfs_avoidance_fear',
       'dfs_specific_stimuli_fear', 'dfs_physiological_arousal'],
      dtype='str')

## superdataset-data.ttl  
The OWL (i.e., Turtle) transformation is built from 4 different files.
* **superdataset.ttl** is the OWL representaiton of the LinkML schema.
  * Run `make superdataset-owl` to build.
* **superdataset-puns.ttl** is the OWL representation of the data properties that have been punned as classes based on the specialized annotations in the LinkML schema.
  * Run `make superdataset-puns.ttl` to build.
* **superdataset-records.ttl** is the raw OWL representation of records in the superdataset CSV.
  * Run `make superdataset-records.ttl` to build.
* **superdataset-data.ttl** is the result of merging *superdataset.ttl*, *superdataset-puns.ttl*, and *superdataset-records.ttl*.
  * Run `make superdataset-data.ttl` to build.

## preview OWL file build from LinkML schema

In [7]:
print(head('../owl/superdataset.ttl', 120, 135))


###  https://w3id.org/OHD/DFA-study/superdataset/Participant_id
superdataset:Participant_id rdf:type owl:DatatypeProperty ;
                            rdfs:range xsd:string ;
                            rdfs:label "Participant_id" ;
                            skos:definition "The participant's study ID." ;
                            skos:inScheme <https://w3id.org/OHD/DFA-study/superdataset> .


###  https://w3id.org/OHD/DFA-study/superdataset/Participant_id_original
superdataset:Participant_id_original rdf:type owl:DatatypeProperty ;
                                     rdfs:range xsd:string ;
                                     rdfs:label "Participant_id_original" ;
                                     skos:definition "The participant's original study ID." ;
                                     skos:inScheme <https://w3id.org/OHD/DFA-study/superdataset> .




### preview punned data properties

In [8]:
print(head('DFA-data/superdataset-puns.ttl', 10, 20))

@prefix NCIT: <http://purl.obolibrary.org/obo/NCIT_> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix dwc: <http://rs.tdwg.org/dwc/terms/> .

superdataset:dfs1 rdf:type owl:Class; rdfs:subClassOf OHD:0000421 . 

superdataset:dfs2 rdf:type owl:Class; rdfs:subClassOf OHD:0000422 . 

superdataset:dfs3 rdf:type owl:Class; rdfs:subClassOf OHD:0000423 . 

superdataset:dfs4 rdf:type owl:Class; rdfs:subClassOf OHD:0000424 . 



### preview RDF records

In [9]:
print(head('DFA-data/superdataset-records.ttl', 4, 10), 
      '\n',
      head('DFA-data/superdataset-records.ttl', 69, 75))

superdataset:1 superdataset:Age 44 ;
    superdataset:Ethnicity_new 1 ;
    superdataset:Last_Dental_Visit 1 ;
    superdataset:Participant_id "1" ;
    superdataset:Participant_id_original "1" ;
    superdataset:Race 1 ;
    superdataset:Sex 2 ;
 
 superdataset:10 superdataset:Age 56 ;
    superdataset:Ethnicity_new 1 ;
    superdataset:Last_Dental_Visit 1 ;
    superdataset:Participant_id "10" ;
    superdataset:Participant_id_original "12" ;
    superdataset:Race 1 ;
    superdataset:Sex 2 ;



## query data  

In [10]:
g = Graph().parse('DFA-data/superdataset-data.ttl')

In [11]:
q = """
PREFIX sd: <https://w3id.org/OHD/DFA-study/superdataset/>
SELECT ?source (COUNT(?score) AS ?n) (AVG(?score) AS ?mean_dfs_total)
WHERE {
  ?r sd:source ?source ;
     sd:dfs_total_score ?score .
  FILTER(?score NOT IN (-9999, -9876))
}
GROUP BY ?source
ORDER BY ?source
"""

In [12]:
sparql_df(q, g)

,source,n,mean_dfs_total
0,bph,164,39.79878
1,cohra1,1488,36.987903
2,cohra2,1478,33.534506
3,f31,102,45.960784
4,randall,552,43.416667
5,tic,466,43.156652


In [13]:
q = """
PREFIX sd: <https://w3id.org/OHD/DFA-study/superdataset/>
SELECT ?source
       (SUM(IF(?score >= 0, 1, 0)) AS ?n_valid)
       (SUM(IF(?score = -9999, 1, 0)) AS ?n_missing)
       (SUM(IF(?score = -9876, 1, 0)) AS ?n_not_asked)
       (AVG(IF(?score >= 0, ?score, ?unbound)) AS ?mean_dfs_total)
WHERE { 
    ?r sd:source ?source ; 
       sd:dfs_total_score ?score 
}
GROUP BY ?source
"""

In [14]:
sparql_df(q, g)

,source,n_valid,n_missing,n_not_asked,mean_dfs_total
0,bph,164,3,0,39.79878
1,tic,466,3,0,43.156652
2,cohra1,1488,156,0,36.987903
3,f31,102,0,0,45.960784
4,randall,552,122,0,43.416667
5,cohra2,1478,6,0,33.534506
6,bwhs,0,0,484,0.0
7,mdas,0,0,72,0.0
8,sohea,0,0,5067,0.0


## spot check sparql query results

In [15]:
bph_df = df[df.source == "bph"]
bph_df.shape

(167, 65)

In [16]:
# check bph dfs total mean
round(bph_df.loc[bph_df.dfs_total_score >=0, 'dfs_total_score'].mean(), 2)

np.float64(39.8)

In [17]:
# check bph missing
assert len(bph_df.loc[bph_df.dfs_total_score < 0, 'dfs_total_score']) == 3

In [18]:
tic_df = df[df.source == "tic"]
tic_df.shape

(469, 65)

In [19]:
# check tic dfs total mean
round(tic_df.loc[tic_df.dfs_total_score >=0, 'dfs_total_score'].mean(), 2)

np.float64(43.16)

In [20]:
# check tic missing
assert len(tic_df.loc[tic_df.dfs_total_score < 0, 'dfs_total_score']) == 3